# EvidenceLab #02
## Missing Data in Real Life
### See the pattern. Before you fill the gap.

**EvidenceLab | See it. Understand it. Run it.**  
**Dr. Amobi Andrew Onovo, PhD, MPH**

Missing values are simply blank spaces in a dataset. The important question is not only **how many are blank**, but **why they may be blank**.

> **Percentage is a warning light, not a diagnosis.**

## What you will do

This lesson has **two hands-on tracks**.

### Track 1 | Learn with simulated data
We create a small fictional dataset where we know the complete answers first. Then we deliberately hide some values in three different ways: **MCAR, MAR and MNAR**.

Because we know what was hidden, we can ask: **How well did each method fill the gaps?**

### Track 2 | Try the same workflow on Framingham data
We then use a cardiovascular-risk teaching dataset associated with Framingham analyses. Here, the real missing values are unknown.

You will:
**see the gaps → investigate the pattern → choose methods → fill missing values → compare results → check whether the conclusion changes.**

> The Framingham CSV used here is a teaching copy, not the full original Framingham Heart Study. This notebook is for learning, not clinical prediction.

## How to use this notebook

Each section explains the idea first, then shows the code that puts it into practice.

1. Read the short explanation for the step.
2. Run the analysis cell.
3. Look at the chart or table.
4. Use the **🔎 What does this mean?** note to connect the output to the EvidenceLab workflow.
5. When you see **⚙️ Supporting code**, run it to prepare reusable functions or calculations used in the next step.

You can focus first on **what each step is doing and what the output means**, then explore the implementation in as much detail as you wish.

For the easiest experience in Colab, choose **Runtime → Run all**.

## The EvidenceLab workflow

**MAP → QUESTION → MECHANISM → METHOD → STRESS-TEST → MODEL → INSIGHTS**


### Setup · install only missing packages

**What:** Make the required Python packages available.

**Why:** Colab images change over time.

**Your turn:** Run this cell. An installation may take a minute; no account or API key is needed.

**Look for:** A completion message; package versions are recorded at the end.

**Interpretation:** The default teaching results use a fixed seed; numerical libraries can introduce small version-dependent differences.


In [ ]:
import sys, subprocess, importlib.util
packages = {'numpy':'numpy', 'pandas':'pandas', 'scipy':'scipy',
            'statsmodels':'statsmodels', 'sklearn':'scikit-learn',
            'matplotlib':'matplotlib', 'plotly':'plotly'}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print('Packages available. Continue below.')


### Optional: use your own Framingham CSV

The simulation works without any file.

For the real-world section, the notebook can use its teaching copy automatically. If you already have `framingham.csv` and want to use it, Colab can open a normal file picker.

The simplest Colab upload is:

```python
from google.colab import files
uploaded = files.upload()
```

No file path is needed.

**For your first run:** keep the defaults below.  
**To use your own file:** change `USE_UPLOAD = False` to `True`, run the cell, and choose your CSV.


In [ ]:
# Choose what you want to run.
RUN_FRAMINGHAM = True   # True = include the real-world lesson
USE_UPLOAD = False      # Change to True if you want the Colab file picker
ALLOW_FALLBACK = True   # Use the teaching copy when no file is uploaded

FRAMINGHAM_UPLOAD_BYTES = None

if USE_UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    file_name = next(iter(uploaded))
    FRAMINGHAM_UPLOAD_BYTES = uploaded[file_name]
    print("Loaded:", file_name)
else:
    print("Ready ✓  The notebook will use the teaching copy for Track 2.")


### ⚙️ Supporting code: load analysis and plotting tools

This cell prepares reusable functions used in the analysis that follows.

**▶ Run the cell to prepare the next step.**


In [ ]:
import os, json, hashlib, inspect, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import plotly.express as px
from scipy.special import expit
from scipy.optimize import brentq
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.imputation.mice import MICEData
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression, BayesianRidge
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score, balanced_accuracy_score
from sklearn.base import clone, BaseEstimator, TransformerMixin
from IPython.display import display, Markdown


from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, brier_score_loss
import urllib.request, io
import plotly.io as pio


### Set reproducible defaults and the EvidenceLab palette

**What:** Set the seed, sample size, MI draws and chart colors.

**Why:** These are the main controls for a reproducible experiment.

**Your turn:** Use defaults first. Later change one value and rerun from here.

**Look for:** All files are written into outputs/ alongside the running notebook.

**Interpretation:** N and M are teaching choices, not universal recommendations. M must be at least two for Rubin pooling.


In [ ]:
SEED, N, M = 20260915, 2500, 20
TARGET_MISSING = 0.18
DELTAS = [-6., -3., 0., 3., 6.]
assert N >= 2000 and M >= 2 and 0 < TARGET_MISSING < 1
OUT = Path('outputs'); FIG = OUT / 'figures'
FIG.mkdir(parents=True, exist_ok=True)
PALETTE = dict(observed='#1565C0', missing='#E83E5B', robust='#00865D',
               methods='#7036DA', caution='#FFDF3D', ink='#092C59')
COLORS = [PALETTE['observed'], PALETTE['robust'], PALETTE['methods'], PALETTE['missing']]
plt.rcParams.update({'figure.figsize':(9,5), 'font.size':13,
                     'axes.spines.top':False, 'axes.spines.right':False})
pio.templates.default = 'plotly_white'


### ⚙️ Supporting code: savefig

This cell prepares reusable functions used in the analysis that follows.

**▶ Run the cell to prepare the next step.**


In [ ]:
def savefig(name):
    plt.savefig(FIG / (name + ".png"), dpi=300, bbox_inches="tight")
    plt.savefig(FIG / (name + ".svg"), bbox_inches="tight")
    plt.show()
    plt.close("all")


### ⚙️ Supporting code: table

This cell prepares reusable functions used in the analysis that follows.

**▶ Run the cell to prepare the next step.**


In [ ]:
def table(df, name):
    df.to_csv(OUT / (name + ".csv"), index=False)
    display(df)


### ⚙️ Supporting code: interactive visuals

This cell prepares the functions used to create the interactive charts and missingness displays that follow.

**▶ Run the cell to prepare the next step.**


In [ ]:
def interactive(fig, name):
    fig.update_layout(font=dict(size=16, color=PALETTE['ink']),
                      margin=dict(l=55,r=30,t=100,b=65), height=540)
    fig.write_html(FIG / (name+'.html'), include_plotlyjs=True)
    fig.show()

def missing_map(data, columns, name, title, rows=160):
    mask = data.loc[:, columns].iloc[:rows].isna().astype(int)
    fig = px.imshow(mask.T, aspect='auto', zmin=0, zmax=1,
                    color_continuous_scale=[[0,PALETTE['observed']],
                     [.499,PALETTE['observed']],[.5,PALETTE['missing']],[1,PALETTE['missing']]],
                    labels=dict(x='Displayed row position', y='Variable', color='Status'), title=title)
    fig.update_coloraxes(colorbar=dict(tickvals=[0,1],ticktext=['Observed','Missing']))
    interactive(fig,name+'_interactive')
    plt.figure(figsize=(11,max(3,len(columns)*.48)))
    plt.imshow(mask.T, aspect='auto', interpolation='nearest',
               cmap=ListedColormap([PALETTE['observed'],PALETTE['missing']]),vmin=0,vmax=1)
    plt.yticks(range(len(columns)),columns)
    plt.xlabel('Row position in displayed subset · blue observed / coral missing')
    plt.title(title); savefig(name)

def method_forest(data, name, title, reference=1):
    fig, ax = plt.subplots(figsize=(10,max(4,len(data)*.55)))
    for i, row in data.reset_index(drop=True).iterrows():
        color=PALETTE['robust'] if row['interval_kind']=='Rubin pooled' else PALETTE['methods']
        ax.plot(row['estimate'],i,'o',color=color,markersize=7)
        if pd.notna(row['low']):
            ax.plot([row['low'],row['high']],[i,i],color=color,linewidth=2)
    ax.axvline(reference,color=PALETTE['ink'],linestyle='--',label=f'Null = {reference}')
    ax.set_yticks(range(len(data)),data['method']); ax.invert_yaxis()
    ax.set_xlabel('Adjusted glucose odds ratio per 10 recorded units')
    ax.set_title(title); ax.legend(); savefig(name)


## Track 1 · The simulation laboratory

Age, income, biomarker, blood pressure and outcome are fictional. The generating biomarker coefficient is **0.8 outcome points per biomarker unit**, conditional on the other predictors. The complete-data sample estimate will differ because sampling varies. Binary smoker and ordered education retain valid categories; region and occupation are nominal.


### Create complete truth

**What:** Generate fictional complete records.

**Why:** Known truth lets us score deliberately hidden values later.

**Your turn:** Run; inspect the first rows. Do not upload anything.

**Look for:** No missing cells and N rows.

**Interpretation:** These values and the generating coefficient are simulation inputs, not real health findings.


In [ ]:
rng = np.random.default_rng(SEED)
age = rng.uniform(25, 80, N)
sex = rng.binomial(1, .48, N)
region = rng.choice(["North", "South", "West"], N)
occupation = rng.choice(["Office", "Service", "Field"], N)
education = rng.choice([1., 2., 3., 4.], N, p=[.18,.32,.30,.20])
smoker = rng.binomial(1, expit(-.8 + .35*sex - .25*(education-2)), N).astype(float)
income = 50 + .3*(age-50) + 5*education + 7*(occupation=="Office") + rng.normal(0, 10, N)
biomarker = 30 + .25*(age-50) + 4*smoker + .08*(income-65) + rng.normal(0, 6, N)
systolic_bp = 118 + .35*(age-50) + 3*sex + 2*smoker + rng.normal(0, 9, N)
outcome = 10 + .8*biomarker + .12*age + 2*sex + 3*smoker + .04*income + .7*education + .05*systolic_bp + 2*(region=="South") + rng.normal(0, 8, N)
truth_df = pd.DataFrame(dict(age=age, income=income, biomarker=biomarker, systolic_bp=systolic_bp, sex=sex, smoker=smoker, occupation=occupation, region=region, education=education, outcome=outcome))
assert len(truth_df) >= 2000 and not truth_df.isna().any().any()
truth_df.to_csv(OUT / "simulation_truth.csv", index=False)
display(truth_df.head())


### ⚙️ Supporting code: define the missingness mechanisms

This cell prepares reusable functions used in the analysis that follows.

**▶ Run the cell to prepare the next step.**


In [ ]:
TARGETS = ["biomarker", "income", "smoker", "education"]
def calibrated_prob(score, target=TARGET_MISSING):
    intercept = brentq(lambda b: expit(b + score).mean() - target, -30, 30)
    return expit(intercept + score)

def introduce_missingness(truth, mechanism, seed):
    r = np.random.default_rng(seed)
    out = truth.copy(deep=True)
    probs = pd.DataFrame(index=truth.index)
    for col in TARGETS:
        if mechanism == "MCAR":
            p = np.full(len(truth), TARGET_MISSING)
        elif mechanism == "MAR":
            # Every driver remains fully observed in every incomplete copy.
            score = .9*(truth.age-50)/15 + .55*truth.sex + .45*(truth.region=="South")
            p = calibrated_prob(score)
        elif mechanism == "MNAR":
            # Depends on the very value subsequently hidden, even conditional on observed variables.
            score = 1.2*(truth[col]-truth[col].mean())/truth[col].std()
            p = calibrated_prob(score)
        else:
            raise ValueError(mechanism)
        out.loc[r.random(len(truth)) < p, col] = np.nan
        probs[col] = p
    return out, probs


### Three simple ideas to remember

**MCAR | Chance → missing**  
Example: a measuring device randomly fails. The failure is unrelated to the person or value being measured.

**MAR | Observed information → missing**  
Example: younger participants are less likely to report income, and **age is recorded**. Something we already know helps explain the gap.

**MNAR | Missing value itself → missing**  
Example: people with very high incomes may be less willing to report income. The value we cannot see may help explain why it is missing.

> In the simulation we know which mechanism is true because **we created it**. In real data, a chart alone cannot prove MAR versus MNAR.


### Hide values three different ways

**What:** Create MCAR, MAR and MNAR copies of the same truth.

**Why:** Holding the original data fixed isolates the hiding rule.

**Your turn:** Run; later change TARGET_MISSING and rerun the notebook.

**Look for:** Three separate incomplete datasets and their known hiding probabilities.

**Interpretation:** The mechanism is known only because we programmed it. Similar percentages need not mean similar mechanisms.


In [ ]:
datasets, probabilities = {}, {}
for i, mechanism in enumerate(["MCAR", "MAR", "MNAR"]):
    datasets[mechanism], probabilities[mechanism] = introduce_missingness(truth_df, mechanism, SEED+i+1)
    datasets[mechanism].to_csv(OUT / f"simulation_{mechanism}.csv", index=False)
    probabilities[mechanism].to_csv(OUT / f"mask_probabilities_{mechanism}.csv", index=False)


## 1 · MAP — what is missing?


### Compare the missing percentages

**What:** Count missing cells for each variable and mechanism.

**Why:** Amount alone cannot diagnose the cause.

**Your turn:** Run; hover over bars to compare exact percentages.

**Look for:** Realized rates vary around the chosen expected rate.

**Interpretation:** No bar height establishes MCAR, MAR or MNAR.


In [ ]:
profile = pd.concat([pd.DataFrame({'Variable':TARGETS,
    'Missing (%)':100*d[TARGETS].isna().mean().values, 'Mechanism':name})
    for name,d in datasets.items()],ignore_index=True)
table(profile,'missingness_profile')
fig=px.bar(profile,x='Variable',y='Missing (%)',color='Mechanism',barmode='group',
    color_discrete_sequence=COLORS,title='Similar amounts, different hiding rules')
interactive(fig,'missing_percentages')


#### 🔎 What does this mean?
Similar missing percentages do **not** mean the missingness happened for the same reason. The mechanism matters.


### Locate the gaps

**What:** Draw the first 160 MAR records.

**Why:** Patterns are easier to investigate when cells remain visible.

**Your turn:** Run; hover to identify cells. Change rows=160 to see a larger subset.

**Look for:** Blue observed cells and coral missing cells.

**Interpretation:** This is a display subset; its appearance does not prove a mechanism.


In [ ]:
missing_map(datasets['MAR'],TARGETS,'simulation_missing_map','MAR simulation · locate the gaps',rows=160)


#### 🔎 What does this mean?
The map helps your eye spot where gaps concentrate. Blue cells are observed; coral cells are missing.


### Look for variables missing together

**What:** Correlate missingness indicators across all simulated MAR records.

**Why:** The full dataset is more informative than the small display subset.

**Your turn:** Run and compare off-diagonal entries.

**Look for:** Positive values indicate co-occurring gaps; the diagonal is always one for nonconstant indicators.

**Interpretation:** Correlation of indicators is an exploratory clue, not a missingness-mechanism test.


In [ ]:
mask_all = datasets['MAR'][TARGETS].isna().astype(int)
co = mask_all.corr()
co.to_csv(OUT/'co_missingness.csv')
fig=px.imshow(co,text_auto='.2f',zmin=-1,zmax=1,
    color_continuous_scale=[PALETTE['missing'],'#FFFFFF',PALETTE['observed']],
    title='Co-missingness · all MAR simulation rows')
interactive(fig,'co_missingness')
display(mask_all.value_counts().head(10).rename('Rows'))


#### 🔎 What does this mean?
Variables that are often missing together may share a collection process or observed predictor. Treat this as a clue, not a diagnosis.


## 2 · QUESTION — is missingness related to something we observed?

Now we ask a simple question:

> **Do people/rows with a missing biomarker look different on information we can still see?**

We compare recorded characteristics such as age, sex and region.

If groups differ, that is a **clue** that missingness is related to observed information.

**Important:** a clue is not proof of MAR or MNAR.


### ⚙️ Supporting code: compare observed groups

This cell prepares reusable functions used in the analysis that follows.

**▶ Run the cell to prepare the next step.**


In [ ]:
def investigate_simulation():
    association_rows=[]
    missingness_models=[]
    for mechanism,d in datasets.items():
        q=d.copy(); q["R_biomarker"] = q.biomarker.isna().astype(int)
        for name in ["age","systolic_bp"]:
            a=q.loc[q.R_biomarker==0,name]; b=q.loc[q.R_biomarker==1,name]
            association_rows.append(dict(mechanism=mechanism,variable=name,observed_group_mean=a.mean(),missing_group_mean=b.mean(),mean_difference=b.mean()-a.mean(),welch_p=stats.ttest_ind(b,a,equal_var=False).pvalue))
        ct=pd.crosstab(q.region,q.R_biomarker)
        ct.to_csv(OUT / f"region_crosstab_{mechanism}.csv")
        fit=smf.logit("R_biomarker ~ age + sex + C(region)",q).fit(disp=False)
        for term in fit.params.index:
            lo,hi=fit.conf_int().loc[term]
            missingness_models.append(dict(mechanism=mechanism,term=term,odds_ratio=np.exp(fit.params[term]),low=np.exp(lo),high=np.exp(hi),p=fit.pvalues[term],model_lr_p=fit.llr_pvalue))
    table(pd.DataFrame(association_rows),"observed_missing_comparisons")
    table(pd.DataFrame(missingness_models),"missingness_logistic_models")
    return pd.DataFrame(association_rows), pd.DataFrame(missingness_models)


### Investigate observed characteristics

**What:** Compare observed age and blood pressure, and fit missingness models.

**Why:** Relationships with recorded information help formulate questions.

**Your turn:** Run. Look at effect sizes and uncertainty, not only p-values.

**Look for:** Mean differences and missingness odds ratios by mechanism.

**Interpretation:** Observed associations are clues. Lack of significance does not establish MCAR; these models cannot prove MAR versus MNAR.


In [ ]:
association_table, missingness_model_table = investigate_simulation()
age_groups=association_table.query("variable == 'age'")
fig=px.scatter(age_groups,x='mean_difference',y='mechanism',
    title='Mean age difference: missing group minus observed group')
fig.add_vline(x=0,line_dash='dash'); interactive(fig,'observed_group_association')


#### 🔎 What does this mean?
Differences between the observed and missing groups tell us where to investigate next. They do not prove MAR or MNAR.


## 3 · MECHANISM → 4 · METHOD

Once we have looked at the pattern, we can decide how to handle the gaps.

### The methods in plain English

| Method | Think of it as... | Main caution |
|---|---|---|
| **Complete case** | Use only rows that have the values you need | Can throw away useful information |
| **Mean / median / mode** | Fill with a simple typical value | Easy, but can distort variation |
| **Regression** | Predict the missing value from other columns | Can look more certain than it really is |
| **KNN** | Borrow from similar rows | Depends on how “similar” is defined |
| **PMM / MICE** | Create several plausible completed datasets | Requires assumptions and more work |

We use the simple methods as **baselines**, not automatic best practice.

Next, the notebook prepares these methods for us. Technical setup cells are collapsed so beginners can focus on the learning.


### ⚙️ Supporting code: model definitions

This cell prepares reusable functions used in the analysis that follows.

**▶ Run the cell to prepare the next step.**


In [ ]:
NUM = ["age","income","biomarker","systolic_bp","sex","outcome"]
CAT_MISSING = ["smoker","education"]
FORMULA = 'outcome ~ biomarker + age + sex + smoker + income + education + systolic_bp + region_South + region_West + occupation_Office + occupation_Service'


### ⚙️ Supporting code: encoded

This cell prepares reusable functions used in the analysis that follows.

**▶ Run the cell to prepare the next step.**


In [ ]:
def encoded(d):
    return pd.get_dummies(d,columns=["region","occupation"],drop_first=True,dtype=float).astype(float)


### ⚙️ Supporting code: analysis

This cell prepares reusable functions used in the analysis that follows.

**▶ Run the cell to prepare the next step.**


In [ ]:
def analysis(d):
    return smf.ols(FORMULA,d).fit()


### ⚙️ Supporting code: point_complete

This cell prepares reusable functions used in the analysis that follows.

**▶ Run the cell to prepare the next step.**


In [ ]:
def point_complete(d,method):
    e=encoded(d); out=e.copy()
    numeric=[c for c in e if c not in CAT_MISSING]
    scaler=StandardScaler(); z=scaler.fit_transform(e[numeric])
    if method in ["Mean","Median"]:
        imputer=SimpleImputer(strategy=method.lower())
    elif method=="Regression":
        imputer=IterativeImputer(estimator=LinearRegression(),max_iter=20,tol=1e-4,random_state=SEED)
    elif method=="KNN":
        imputer=KNNImputer(n_neighbors=7,weights="distance")
    else: raise ValueError(method)
    out[numeric]=scaler.inverse_transform(imputer.fit_transform(z))
    features=StandardScaler().fit_transform(out[numeric])
    for c in CAT_MISSING:
        obs=e[c].notna(); miss=~obs
        if method in ["Mean","Median"]:
            out.loc[miss,c]=e.loc[obs,c].mode().iloc[0]
        else:
            classifier=LogisticRegression(max_iter=1500) if method=="Regression" else KNeighborsClassifier(n_neighbors=15,weights="distance")
            classifier.fit(features[obs],e.loc[obs,c]); out.loc[miss,c]=classifier.predict(features[miss])
    assert not out.isna().any().any()
    return out


### ⚙️ Supporting code: mice_complete

This cell prepares reusable functions used in the analysis that follows.

**▶ Run the cell to prepare the next step.**


In [ ]:
def mice_complete(d, seed, m=M):
    e=encoded(d)
    np.random.seed(seed)  # compatibility with statsmodels versions before rng argument
    kw={"rng":seed} if "rng" in inspect.signature(MICEData).parameters else {}
    imp=MICEData(e,perturbation_method="gaussian",k_pmm=5,**kw)
    imp.set_imputer("smoker", model_class=sm.Logit, fit_kwds={"disp":False}, k_pmm=5)
    trace=[]
    imp.update_all(10)
    completed=[]
    for i in range(m):
        imp.update_all(3)
        filled=imp.data.copy(deep=True)
        for c in TARGETS:
            observed=e[c].notna()
            assert np.array_equal(filled.loc[observed,c],e.loc[observed,c])
            assert np.isin(filled.loc[~observed,c],e.loc[observed,c]).all()
        completed.append(filled)
        trace.append({"draw":i+1,"mean_biomarker_imputed":filled.loc[e.biomarker.isna(),"biomarker"].mean(),"coefficient":analysis(filled).params["biomarker"]})
    return completed,pd.DataFrame(trace)


### Create baseline and stochastic completed datasets

**What:** Run mean, median, regression, KNN and PMM/MICE.

**Why:** Different methods make different assumptions about the unseen values.

**Your turn:** Run once; the progress messages may take a few minutes.

**Look for:** Each mechanism gets four single completions and M stochastic PMM completions.

**Interpretation:** Inference may use the recorded outcome in the imputation model. Prediction later starts again from raw predictors and excludes test outcomes.


In [ ]:
completed_by_mechanism={}; point_by_mechanism={}; traces={}
for i,(mechanism,d) in enumerate(datasets.items()):
    point_by_mechanism[mechanism]={method:point_complete(d,method) for method in ["Mean","Median","Regression","KNN"]}
    completed_by_mechanism[mechanism],traces[mechanism]=mice_complete(d,SEED+100+i)
    traces[mechanism].to_csv(OUT/f"mice_trace_{mechanism}.csv",index=False)
    print(mechanism, "completed", M, "PMM/MICE datasets")


### The EvidenceLab Multi-Method Triangulation Check

Instead of trusting one method automatically, we compare several reasonable approaches.

We ask four simple questions:

1. **Plausibility:** Do the filled values make sense?
2. **Distribution:** Do they preserve a believable spread?
3. **Known-truth error:** In the simulation, how close are they to the values we deliberately hid?
4. **Downstream result:** Does the final analysis change?

> **Agreement strengthens robustness. Disagreement is information.**

A lower RMSE means closer individual fills in this simulation. It does **not** automatically mean a method gives the best statistical inference.


### ⚙️ Supporting code: score only deliberately hidden cells

This cell prepares reusable functions used in the analysis that follows.

**▶ Run the cell to prepare the next step.**


In [ ]:
def score_simulation():
    metric_rows=[]; plausibility=[]; distribution_rows=[]
    for mechanism,d in datasets.items():
        candidates={k:[v] for k,v in point_by_mechanism[mechanism].items()}
        candidates["PMM/MICE"]=completed_by_mechanism[mechanism]
        for method,draws in candidates.items():
            for col in TARGETS:
                mask=d[col].isna(); actual=truth_df.loc[mask,col].to_numpy()
                vals=np.array([x.loc[mask,col].to_numpy() for x in draws])
                row=dict(mechanism=mechanism,method=method,variable=col,n_hidden=int(mask.sum()))
                if col in CAT_MISSING:
                    row.update(accuracy=float(np.mean([accuracy_score(actual,v) for v in vals])),balanced_accuracy=float(np.mean([balanced_accuracy_score(actual,v) for v in vals])))
                    if col=="education": row["ordinal_MAE"]=float(np.mean(np.abs(vals-actual)))
                else:
                    row.update(MAE=float(np.mean(np.abs(vals-actual))),RMSE=float(np.mean(np.sqrt(np.mean((vals-actual)**2,axis=1)))))
                metric_rows.append(row)
                plausibility.append(dict(mechanism=mechanism,method=method,variable=col,min_imputed=float(vals.min()),max_imputed=float(vals.max()),outside_observed_range=int(((vals<d[col].min())|(vals>d[col].max())).sum()),invalid_category=int((~np.isin(vals,truth_df[col].unique())).sum()) if col in CAT_MISSING else 0))
                distribution_rows.append(dict(mechanism=mechanism,method=method,variable=col,true_hidden_mean=float(actual.mean()),true_hidden_sd=float(actual.std(ddof=1)),imputed_mean=float(vals.mean()),mean_within_draw_sd=float(np.mean(np.std(vals,axis=1,ddof=1)))))
    metrics=pd.DataFrame(metric_rows); table(metrics,"triangulation_metrics")
    table(pd.DataFrame(plausibility),"plausibility_checks")
    table(pd.DataFrame(distribution_rows),"distribution_summary")
    return metrics, pd.DataFrame(plausibility), pd.DataFrame(distribution_rows)


### Compare error and plausibility

**What:** Score the hidden numeric and categorical values.

**Why:** Simulation supplies truth; naturally missing Framingham values do not.

**Your turn:** Run. Filter the table by variable or method.

**Look for:** Numeric MAE/RMSE, valid category checks and distribution summaries.

**Interpretation:** PMM metrics average errors across individual draws; they are not error after averaging fills. Lowest RMSE does not automatically mean best inference.


In [ ]:
metrics, plausibility, distribution_summary = score_simulation()
biomarker_metrics=metrics.query("variable == 'biomarker'")
fig=px.bar(biomarker_metrics,x='method',y='RMSE',color='mechanism',barmode='group',
    color_discrete_sequence=COLORS,title='Known-truth error · one simulated dataset')
interactive(fig,'simulation_rmse')


#### 🔎 What does this mean?
Use several criteria, not one score. A method can fill individual values closely but still handle uncertainty poorly.


### Compare distributions, not only error

**What:** Overlay hidden truth and completed biomarker distributions.

**Why:** A conditional point prediction can suppress variation.

**Your turn:** Run and compare spread as well as center.

**Look for:** PMM draw 1 is explicitly labeled as one draw.

**Interpretation:** A plausible-looking distribution does not establish correct inference.


In [ ]:
mask=datasets["MAR"].biomarker.isna()
bins=np.linspace(truth_df.biomarker.min()-5,truth_df.biomarker.max()+5,30)
plt.hist(truth_df.loc[mask,"biomarker"],bins=bins,density=True,histtype="step",linewidth=2,label="Hidden truth",color=PALETTE["ink"])
for color,method in zip(COLORS,["Regression","KNN","PMM/MICE"]):
    values=completed_by_mechanism["MAR"][0] if method=="PMM/MICE" else point_by_mechanism["MAR"][method]
    plt.hist(values.loc[mask,"biomarker"],bins=bins,density=True,histtype="step",linewidth=2,label=method+ (" (draw 1)" if method=="PMM/MICE" else ""),color=color)
plt.xlabel("Biomarker (arbitrary units)"); plt.ylabel("Density"); plt.legend(); plt.title("MAR: compare distributions as well as point error")
savefig("imputed_distributions")


### Proper multiple imputation: why create more than one completed dataset?

A single filled dataset acts as if we know the replacements with certainty. We do not.

Multiple imputation creates **several plausible completed datasets**.

Think of it as:

**20 plausible datasets → run the same analysis 20 times → combine the results**

The combining step is called **Rubin pooling**.

Why do this? It lets the final uncertainty reflect both:
- normal sampling uncertainty, and
- uncertainty about the missing values.

You do not need to memorize the pooling formula in this beginner lesson. The technical engine handles it.


### ⚙️ Supporting code: rubin_pool

This cell prepares reusable functions used in the analysis that follows.

**▶ Run the cell to prepare the next step.**


In [ ]:
def rubin_pool(fits):
    q=np.array([f.params.to_numpy() for f in fits]); u=np.array([f.cov_params().to_numpy() for f in fits])
    m=len(fits); assert m>=2
    qbar=q.mean(axis=0); ubar=u.mean(axis=0); between=np.cov(q,rowvar=False,ddof=1)
    total=ubar+(1+1/m)*between
    within_diag=np.diag(ubar); between_diag=np.diag(between); total_diag=np.diag(total)
    lam=(1+1/m)*between_diag/total_diag
    old_df=np.where(lam>1e-12,(m-1)/np.maximum(lam,1e-12)**2,np.inf)
    complete_df=float(np.mean([f.df_resid for f in fits]))
    observed_df=((complete_df+1)/(complete_df+3))*complete_df*(1-lam)
    df=1/(1/old_df+1/observed_df)
    se=np.sqrt(total_diag); critical=stats.t.ppf(.975,df)
    result=pd.DataFrame(dict(term=fits[0].params.index,estimate=qbar,within=within_diag,between=between_diag,total=total_diag,se=se,df=df,low=qbar-critical*se,high=qbar+critical*se,mcse_mean=np.sqrt(between_diag/m)))
    assert np.allclose(total,ubar+(1+1/m)*between)
    assert np.all(np.linalg.eigvalsh(total)>0)
    return result,total


### Fit each dataset and pool uncertainty

**What:** Fit M copies of the same substantive model, then apply Rubin pooling.

**Why:** Within- and between-imputation uncertainty both matter.

**Your turn:** Run; inspect the biomarker row and trace plot.

**Look for:** Different draws, nonnegative between-imputation variance, and a pooled interval.

**Interpretation:** Repeated deterministic fills are not proper MI. Trace stability alone does not establish convergence or model adequacy.


In [ ]:
pooled={}; fit_sets={}
for mechanism,draws in completed_by_mechanism.items():
    fit_sets[mechanism]=[analysis(x) for x in draws]
    np.savez_compressed(OUT/f"mi_analysis_draws_{mechanism}.npz",estimates=np.array([f.params.to_numpy() for f in fit_sets[mechanism]]),covariances=np.array([f.cov_params().to_numpy() for f in fit_sets[mechanism]]),completed=np.array([x.to_numpy() for x in draws]),columns=np.array(draws[0].columns,dtype=str))
    pooled[mechanism],cov=rubin_pool(fit_sets[mechanism])
    table(pooled[mechanism],f"rubin_pool_{mechanism}")
    pd.DataFrame(cov,index=pooled[mechanism].term,columns=pooled[mechanism].term).to_csv(OUT/f"pooled_covariance_{mechanism}.csv")
for name,trace in traces.items():
    plt.plot(trace.draw,trace.coefficient,"o-",label=name)
plt.xlabel("Retained MICE draw (3 cycles apart)"); plt.ylabel("Biomarker coefficient"); plt.legend()
plt.title("Trace diagnostic: inspect stability; a trace alone cannot prove convergence")
savefig("mice_traces")
truth_fit=analysis(encoded(truth_df))
print("Generating coefficient:",.8,"; complete-data sample estimate:",truth_fit.params["biomarker"])


## 5 · STRESS-TEST — would the conclusion still hold?

Now we deliberately challenge our assumption.

Suppose the missing biomarker values were actually a little **lower** or **higher** than our imputation assumed.

We test five simple scenarios:

**Lower ← −6 | −3 | MAR (0) | +3 | +6 → Higher**

For each scenario we refill the missing biomarker values, rerun the same analysis, and compare the conclusion.

> **The goal is not to guess the perfect scenario. The goal is to see whether the conclusion is fragile or stable.**


### ⚙️ Supporting code: shift only imputed values, refit and pool

This cell prepares reusable functions used in the analysis that follows.

**▶ Run the cell to prepare the next step.**


In [ ]:
def stress_test_simulation():
    forest=[]
    for method,fit in [("Complete truth",truth_fit),("Complete case",analysis(encoded(datasets["MAR"]).dropna()))]:
        lo,hi=fit.conf_int().loc["biomarker"]
        forest.append(dict(method=method,estimate=fit.params["biomarker"],low=lo,high=hi,n=fit.nobs))
    for method,filled in point_by_mechanism["MAR"].items():
        fit=analysis(filled)
        forest.append(dict(method=method+" (point only)",estimate=fit.params["biomarker"],low=np.nan,high=np.nan,n=fit.nobs))
    sensitivity=[]
    mask=datasets["MAR"].biomarker.isna()
    for delta in DELTAS:
        fits=[]
        for original in completed_by_mechanism["MAR"]:
            adjusted=original.copy(deep=True)
            adjusted.loc[mask,"biomarker"]+=delta
            assert np.array_equal(adjusted.loc[~mask,"biomarker"],original.loc[~mask,"biomarker"])
            fits.append(analysis(adjusted))
        p,_=rubin_pool(fits); row=p.set_index("term").loc["biomarker"].to_dict(); row["delta"]=delta
        sensitivity.append(row)
        forest.append(dict(method="MI MAR" if delta==0 else f"MI delta {delta:+g}",estimate=row["estimate"],low=row["low"],high=row["high"],n=N))
    sensitivity=pd.DataFrame(sensitivity); table(sensitivity,"mnar_delta_sensitivity")
    forest=pd.DataFrame(forest); table(forest,"coefficient_comparison")
    fig,ax=plt.subplots(figsize=(9,6))
    for i,r in forest.iterrows():
        ax.plot(r.estimate,i,"o",color=COLORS[1] if r.method.startswith("MI") else COLORS[0])
        if pd.notna(r.low): ax.plot([r.low,r.high],[i,i],color=COLORS[1] if r.method.startswith("MI") else COLORS[0],linewidth=2)
    ax.axvline(.8,color=PALETTE["ink"],ls="--",label="Generating coefficient 0.8")
    ax.axvline(0,color="gray",linewidth=.7); ax.set_yticks(range(len(forest)),forest.method); ax.invert_yaxis()
    ax.set_xlabel("Outcome points per biomarker unit; intervals where justified"); ax.legend()
    ax.set_title("One-variable sensitivity scenarios under the MAR example"); savefig("sensitivity_forest")
    positive=(sensitivity.low>0).all()
    print("All tested delta-scenario 95% intervals remain above zero:",bool(positive))
    print("Estimate range:",sensitivity.estimate.min(),sensitivity.estimate.max())
    print("This statement applies only to the scenarios tested, not all MNAR mechanisms.")
    # Across all known mechanisms, compare point estimates without claiming a repeated-study bias assessment.
    downstream=[]
    for mechanism in datasets:
        for method,filled in point_by_mechanism[mechanism].items():
            downstream.append(dict(mechanism=mechanism,method=method,estimate=analysis(filled).params["biomarker"]))
        downstream.append(dict(mechanism=mechanism,method="PMM/MICE pooled",estimate=pooled[mechanism].set_index("term").loc["biomarker","estimate"]))
    table(pd.DataFrame(downstream),"downstream_by_mechanism")
    return sensitivity, forest, positive, pd.DataFrame(downstream)


### Test lower and higher unseen values

**What:** Apply the chosen deltas to imputed biomarker entries only.

**Why:** Different plausible unseen values can change the same analysis.

**Your turn:** Run. Later widen DELTAS and rerun this step.

**Look for:** Point estimates and intervals under each stated scenario.

**Interpretation:** An unchanged conclusion supports robustness only across those scenarios; it does not prove MAR or rule out all MNAR processes.


In [ ]:
sensitivity, forest, positive, downstream = stress_test_simulation()


#### 🔎 What does this mean?
If the conclusion stays similar as the assumption changes, it is more robust to the scenarios tested.


## 6 · MODEL — machine learning without leakage

This is the one rule to remember:

❌ **Wrong:** fill missing values using the whole dataset → then split

✅ **Right:** split first → learn the imputation from training data → apply it to test data

Why?

The test set is supposed to represent **new, unseen data**. If it helps teach the imputer, information has leaked into model development.

> **The test set must not teach the imputer.**

The technical cells below build the safe pipeline for you.


### ⚙️ Supporting code: prediction feature lists

This cell prepares reusable functions used in the analysis that follows.

**▶ Run the cell to prepare the next step.**


In [ ]:
predictors=truth_df.columns.drop("outcome").tolist()
num_features=["age","income","biomarker","systolic_bp"]
cat_features=["sex","smoker","education","occupation","region"]


### ⚙️ Supporting code: TrainingDonorPMM

This cell prepares reusable functions used in the analysis that follows.

**▶ Run the cell to prepare the next step.**


In [ ]:
class TrainingDonorPMM(TransformerMixin, BaseEstimator):
    """Prediction benchmark: fitted regressions and donors come only from training rows.
    This is single stochastic donor filling, not Rubin-style multiple inference.
    """
    def __init__(self,n_neighbors=5,random_state=SEED):
        self.n_neighbors=n_neighbors; self.random_state=random_state
    def fit(self,X,y=None):
        X=np.asarray(X,dtype=float)
        self.n_features_in_=X.shape[1]; self.training_n_=len(X)
        self.initial_=SimpleImputer(strategy="median").fit(X)
        filled=self.initial_.transform(X); self.models_={}; self.donors_={}
        for j in range(X.shape[1]):
            obs=~np.isnan(X[:,j]); other=np.arange(X.shape[1])!=j
            model=BayesianRidge().fit(filled[obs][:,other],X[obs,j])
            self.models_[j]=model
            self.donors_[j]=(model.predict(filled[obs][:,other]),X[obs,j].copy())
        return self
    def transform(self,X):
        X=np.asarray(X,dtype=float); filled=self.initial_.transform(X)
        r=np.random.default_rng(self.random_state)
        for j,model in self.models_.items():
            miss=np.isnan(X[:,j]); other=np.arange(X.shape[1])!=j
            if not miss.any(): continue
            predicted=model.predict(filled[miss][:,other]); donor_means,donor_values=self.donors_[j]
            k=min(self.n_neighbors,len(donor_means))
            candidates=np.argpartition(np.abs(predicted[:,None]-donor_means[None,:]),k-1,axis=1)[:,:k]
            selected=candidates[np.arange(len(predicted)),r.integers(0,k,len(predicted))]
            filled[miss,j]=donor_values[selected]
        return filled


### ⚙️ Supporting code: make_pipeline

This cell prepares reusable functions used in the analysis that follows.

**▶ Run the cell to prepare the next step.**


In [ ]:
def make_pipeline(imputer):
    pre=ColumnTransformer([
        ("numeric",Pipeline([("scale",StandardScaler()),("impute",clone(imputer))]),num_features),
        ("categorical",Pipeline([("impute",SimpleImputer(strategy="most_frequent")),("encode",OneHotEncoder(handle_unknown="ignore",sparse_output=False))]),cat_features)
    ])
    return Pipeline([("preprocess",pre),("model",LinearRegression())])


### ⚙️ Supporting code: four prespecified predictive approaches

This cell prepares reusable functions used in the analysis that follows.

**▶ Run the cell to prepare the next step.**


In [ ]:
numeric_imputers={"Simple median":SimpleImputer(strategy="median"),"KNN":KNNImputer(n_neighbors=7,weights="distance"),"Iterative regression":IterativeImputer(estimator=BayesianRidge(),random_state=SEED,max_iter=20,tol=1e-3),"PMM donor benchmark":TrainingDonorPMM()}


### ⚙️ Supporting code: evaluate training CV and an untouched test set

This cell prepares reusable functions used in the analysis that follows.

**▶ Run the cell to prepare the next step.**


In [ ]:
def evaluate_simulation_prediction():
    ml_rows=[]; leakage_audits=[]
    for mechanism,d in datasets.items():
        X=d[predictors]; y=d.outcome
        Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=SEED)
        assert set(Xtr.index).isdisjoint(Xte.index)
        cv=KFold(n_splits=4,shuffle=True,random_state=SEED)
        splits=list(cv.split(Xtr,ytr))
        for method,imputer in numeric_imputers.items():
            pipe=make_pipeline(imputer)
            result=cross_validate(pipe,Xtr,ytr,cv=splits,scoring="neg_root_mean_squared_error",return_estimator=True)
            for (train_i,valid_i),fold_pipe in zip(splits,result["estimator"]):
                fitted_scaler=fold_pipe.named_steps["preprocess"].named_transformers_["numeric"].named_steps["scale"]
                assert np.allclose(fitted_scaler.mean_,Xtr.iloc[train_i][num_features].mean())
                assert set(Xtr.iloc[train_i].index).isdisjoint(Xtr.iloc[valid_i].index)
            pipe.fit(Xtr,ytr)
            state=pipe.named_steps["preprocess"].named_transformers_["numeric"]
            before=state.named_steps["scale"].mean_.copy()
            # Test-data mutation must not change learned preprocessing.
            altered=Xte.copy(); altered["biomarker"]=1e6
            pipe.predict(altered)
            assert np.array_equal(before,state.named_steps["scale"].mean_)
            assert np.allclose(before,Xtr[num_features].mean())
            if method=="Simple median":
                train_z=state.named_steps["scale"].transform(Xtr[num_features])
                assert np.allclose(state.named_steps["impute"].statistics_,np.nanmedian(train_z,axis=0))
            elif method=="KNN":
                assert state.named_steps["impute"]._fit_X.shape[0]==len(Xtr)
            elif method=="PMM donor benchmark":
                assert state.named_steps["impute"].training_n_==len(Xtr)
                train_z=state.named_steps["scale"].transform(Xtr[num_features])
                for j,(_,donor_values) in state.named_steps["impute"].donors_.items():
                    assert np.array_equal(donor_values,train_z[~np.isnan(train_z[:,j]),j])
            predicted=pipe.predict(Xte)
            ml_rows.append(dict(mechanism=mechanism,method=method,cv_RMSE=float(-result["test_score"].mean()),cv_fold_SD=float(result["test_score"].std(ddof=1)),test_RMSE=float(np.sqrt(mean_squared_error(yte,predicted))),test_MAE=float(mean_absolute_error(yte,predicted)),train_n=len(Xtr),test_n=len(Xte)))
            leakage_audits.append(dict(mechanism=mechanism,method=method,fold_training_means_verified=True,test_mutation_state_unchanged=True,outcome_excluded=True))
    ml_results=pd.DataFrame(ml_rows); table(ml_results,"ml_prediction")
    table(pd.DataFrame(leakage_audits),"leakage_audit")
    print("Training-CV selections:")
    display(ml_results.loc[ml_results.groupby("mechanism").cv_RMSE.idxmin()])
    return ml_results, pd.DataFrame(leakage_audits)


### Run leakage-safe prediction

**What:** Fit preprocessing inside each training fold and score held-out predictions.

**Why:** The test set must not teach the imputer.

**Your turn:** Run. Compare training-CV RMSE before reading test results.

**Look for:** Audit assertions verify fold-specific training means and unchanged state after a test-data mutation.

**Interpretation:** These candidates were prespecified. Do not retune after viewing test results; this benchmark does not establish clinical performance.


In [ ]:
ml_results, leakage_audits = evaluate_simulation_prediction()


## 7 · INSIGHTS — pause before the real-world track

You have compared **plausibility → distribution → known-truth error → downstream result**. Proper MI pooled uncertainty; the Triangulation Check compared defensible approaches. They are different tasks.

**Try it:** change the seed, expected missing fraction or delta range. State your prediction first, rerun, then explain any disagreement. One run is not a study of repeated-sampling bias or interval coverage.


## Track 2 · Framingham: now try it on real-world data

The simulation was our practice laboratory because we knew the hidden truth.

Now we use Framingham teaching data, where the naturally missing glucose values are **actually unknown**.

This time we cannot ask, “Did we recover the exact hidden value?”

Instead we ask:

1. Where are the gaps?
2. Are the gaps associated with recorded information?
3. What assumption are we making?
4. What happens if we fill the gaps in different reasonable ways?
5. Does the final conclusion change?

The teaching copy used by the notebook contains **388 missing glucose values among 4,240 rows (9.15%)**. That percentage describes the file. It does not tell us whether the missingness is MCAR, MAR or MNAR.


### ⚙️ Supporting code: validated upload, local and pinned-fallback loading

This cell prepares reusable functions used in the analysis that follows.

**▶ Run the cell to prepare the next step.**


In [ ]:
FRAMINGHAM_URL = 'https://raw.githubusercontent.com/GauravPadawe/Framingham-Heart-Study/43cda49873de043c4f155e7cde17d6be9a196d0a/framingham.csv'
FALLBACK_SHA256 = '2c0e57dc0361b420becf1facec0a054af06c420eae0ae2faf0fdc8591fadb018'
FR_REQUIRED = ['age','male','sysBP','diaBP','BMI','heartRate','diabetes',
               'currentSmoker','glucose','TenYearCHD']

def parse_framingham(payload, source):
    data = pd.read_csv(io.BytesIO(payload), na_values=['NA','?',''])
    data.columns = data.columns.str.strip()
    missing_columns = set(FR_REQUIRED)-set(data.columns)
    if missing_columns:
        raise ValueError(f'Missing required columns: {sorted(missing_columns)}')
    for column in FR_REQUIRED:
        data[column] = pd.to_numeric(data[column], errors='raise')
        if not np.isfinite(data[column].dropna()).all():
            raise ValueError(f'{column} contains an infinite value.')
    for column in ['male','diabetes','currentSmoker','TenYearCHD']:
        if not data[column].dropna().isin([0,1]).all():
            raise ValueError(f'{column} must contain only 0, 1 or missing.')
    if len(data)==0 or data['glucose'].notna().sum()<10:
        raise ValueError('This lesson needs nonempty data and at least ten observed glucose values.')
    return data, {'status':'loaded','source':source,
                 'sha256':hashlib.sha256(payload).hexdigest(),
                 'rows':len(data),'columns':len(data.columns),
                 'glucose_missing':int(data.glucose.isna().sum()),
                 'glucose_missing_pct':float(100*data.glucose.isna().mean())}

def load_framingham():
    if not RUN_FRAMINGHAM:
        return None, {'status':'skipped by learner'}
    if FRAMINGHAM_UPLOAD_BYTES is not None:
        return parse_framingham(FRAMINGHAM_UPLOAD_BYTES,'user upload')
    for candidate in [Path('framingham.csv'),Path('data/framingham.csv')]:
        if candidate.is_file():
            return parse_framingham(candidate.read_bytes(),'local companion CSV')
    if not ALLOW_FALLBACK:
        return None, {'status':'no upload/local file; fallback disabled'}
    try:
        with urllib.request.urlopen(FRAMINGHAM_URL,timeout=30) as response:
            payload=response.read()
    except (OSError, TimeoutError) as error:
        print('Teaching mirror unavailable. Track 1 remains usable. Upload a CSV and rerun Track 2.')
        return None, {'status':'fallback unavailable','reason':str(error)}
    if hashlib.sha256(payload).hexdigest()!=FALLBACK_SHA256:
        raise ValueError('Pinned mirror checksum changed; upload the trusted CSV instead.')
    return parse_framingham(payload,'pinned hash-checked teaching mirror')


### Load and check the real-world data

**What:** Use your upload, the companion CSV or the pinned mirror.

**Why:** Reproducible provenance starts before cleaning or imputation.

**Your turn:** Run. No path is needed for the Colab upload route.

**Look for:** The route, rows, columns and glucose missing count.

**Interpretation:** Different source files may have different counts. Missing percentage alone cannot establish a mechanism.


In [ ]:
framingham, framingham_summary = load_framingham()
display(framingham_summary)
if framingham is not None:
    display(framingham.head())
    print(f"Glucose missing: {framingham.glucose.isna().sum():,} / {len(framingham):,} "
          f"({100*framingham.glucose.isna().mean():.2f}%)")


#### 🔎 What does this mean?
You have now loaded the real-world teaching data and measured its missingness from the file itself.


## 1 · MAP — quantify natural missingness


### See the amount and pattern

**What:** Measure natural missingness before replacing anything.

**Why:** Counts describe the task; a map suggests questions.

**Your turn:** Run; hover over a bar and compare it with the map.

**Look for:** Only affected variables appear in the bar chart; the map displays a subset.

**Interpretation:** A visually organized pattern is not proof of MAR or MNAR.


In [ ]:
if framingham is not None:
    fmiss=pd.DataFrame({'variable':framingham.columns,
        'missing_n':framingham.isna().sum().values,
        'missing_pct':100*framingham.isna().mean().values})
    table(fmiss,'framingham_missingness')
    affected=fmiss.query('missing_n > 0').sort_values('missing_pct')
    fig=px.bar(affected,x='missing_pct',y='variable',orientation='h',
        color_discrete_sequence=[PALETTE['missing']],
        title='Framingham · missing values by variable',labels={'missing_pct':'Missing (%)'})
    interactive(fig,'framingham_missing_percentages')
    missing_map(framingham,affected.variable.tolist(),'framingham_missing_map',
                'Framingham · affected variables, first 250 records',rows=250)


#### 🔎 What does this mean?
Framingham has natural gaps. We can see and quantify them, but the picture alone cannot tell us why they occurred.


## 2 · QUESTION — who has unrecorded glucose?


### ⚙️ Supporting code: investigate observed predictors of glucose missingness

This cell prepares reusable functions used in the analysis that follows.

**▶ Run the cell to prepare the next step.**


In [ ]:
def investigate_framingham(data):
    q=data.copy(); q['R_glucose']=q.glucose.isna().astype(int)
    predictors=['age','male','diabetes','currentSmoker','sysBP']
    diagnostic=q[['R_glucose']+predictors].dropna()
    groups=q.groupby('R_glucose')[predictors].agg(['mean','count'])
    groups.to_csv(OUT/'framingham_observed_groups.csv'); display(groups)
    if diagnostic.R_glucose.nunique()<2:
        print('No variation in glucose missingness: a missingness model cannot be fitted.')
        return None
    model=smf.glm('R_glucose ~ age + male + diabetes + currentSmoker + sysBP',
                  diagnostic,family=sm.families.Binomial()).fit()
    ci=model.conf_int()
    odds=pd.DataFrame({'term':model.params.index,'OR':np.exp(model.params.values),
         'low':np.exp(ci[0].values),'high':np.exp(ci[1].values)})
    table(odds,'framingham_missingness_associations')
    print(f'Diagnostic complete-predictor sample: {len(diagnostic):,} / {len(data):,}')
    ages=q[['age','R_glucose']].copy()
    ages['Glucose status']=ages.R_glucose.map({0:'Observed',1:'Missing'})
    fig=px.violin(ages,x='Glucose status',y='age',color='Glucose status',box=True,
        color_discrete_map={'Observed':PALETTE['observed'],'Missing':PALETTE['missing']},
        title='Recorded age by glucose recording status')
    interactive(fig,'framingham_observed_age')
    return odds


### Compare recorded groups

**What:** Compare age and other recorded characteristics by glucose missingness.

**Why:** This investigates observed information without pretending to see the missing glucose.

**Your turn:** Run. Read the analysis sample size and intervals.

**Look for:** An exploratory missingness model and an age-distribution comparison.

**Interpretation:** Associations are clues, not proof of MAR/MNAR. Diagnostic complete-predictor exclusions can also affect the comparison.


In [ ]:
if framingham is not None:
    framingham_associations=investigate_framingham(framingham)


#### 🔎 What does this mean?
Observed characteristics can help explain who has missing glucose. They are evidence to investigate, not proof of a missingness mechanism.


## 3 · MECHANISM — state the assumption before filling anything

For the main Framingham exercise, we use a **working MAR assumption**:

> After using the recorded information in our imputation model, we assume the chance that glucose is missing does not additionally depend on the unrecorded glucose value itself.

That is an **assumption**, not something the dataset proved.

We will later challenge it by asking:

> **What if the missing glucose values were systematically lower or higher than our MAR-based fills?**

## 4 · METHOD — fill the real-world gaps

We compare four practical approaches:

**Median → KNN → Iterative regression → PMM/MICE**

For the first three, we create one completed dataset. PMM/MICE creates several plausible completed datasets so uncertainty can be pooled.

The goal is not to declare a universal winner. It is to see whether reasonable choices tell a similar story.


### ⚙️ Supporting code: define one analytic cohort and one model

This cell prepares reusable functions used in the analysis that follows.

**▶ Run the cell to prepare the next step.**


In [ ]:
FR_CORE=['TenYearCHD','age','male','sysBP','diabetes']
FR_FEATURES=FR_CORE+['diaBP','currentSmoker','BMI','heartRate','glucose']
FR_FORMULA='TenYearCHD ~ glucose10 + age + male + sysBP + diabetes'

def prepare_framingham(data):
    # Complete binary auxiliaries rather than numerically average category labels.
    required=FR_CORE+['currentSmoker']
    base=data.dropna(subset=required)[FR_FEATURES].copy().reset_index(drop=True)
    if len(base)<100 or base.TenYearCHD.nunique()!=2:
        raise ValueError('The teaching model needs at least 100 eligible rows and both outcome classes.')
    if base.notna().sum().min()<10:
        raise ValueError('Each imputation variable needs at least ten observed values.')
    print(f'Common inference cohort: {len(base):,} / {len(data):,}; excluded {len(data)-len(base):,}')
    return base

def fit_framingham(data):
    model_data=data.copy()
    model_data['glucose10']=model_data.glucose/10
    return smf.glm(FR_FORMULA,model_data,family=sm.families.Binomial()).fit()

def framingham_point_completions(base):
    imputers={'Median':SimpleImputer(strategy='median'),
              'KNN':KNNImputer(n_neighbors=5,weights='distance'),
              'Iterative':IterativeImputer(estimator=BayesianRidge(),random_state=SEED,
                                          max_iter=30,tol=1e-3)}
    scaler=StandardScaler().fit(base)
    scaled=scaler.transform(base)
    completed={}
    for name,imputer in imputers.items():
        values=scaler.inverse_transform(imputer.fit_transform(scaled))
        filled=pd.DataFrame(values,columns=base.columns,index=base.index)
        # Restore observed values exactly; only missing entries can change.
        filled=filled.where(base.isna(),base)
        assert filled.notna().all().all()
        for column in base:
            observed=base[column].notna()
            assert np.array_equal(filled.loc[observed,column],base.loc[observed,column])
        completed[name]=filled
    return completed

def framingham_mi(base,m=M):
    np.random.seed(SEED+500)
    kw={'rng':SEED+500} if 'rng' in inspect.signature(MICEData).parameters else {}
    sampler=MICEData(base,perturbation_method='gaussian',k_pmm=5,**kw)
    sampler.update_all(10)
    draws=[]; trace=[]
    for i in range(m):
        sampler.update_all(3)
        filled=sampler.data.copy(deep=True)
        for column in base:
            observed=base[column].notna()
            assert np.array_equal(filled.loc[observed,column],base.loc[observed,column])
            assert filled.loc[~observed,column].isin(base.loc[observed,column]).all()
        draws.append(filled)
        trace.append({'draw':i+1,'glucose_coefficient':fit_framingham(filled).params['glucose10']})
    return draws,pd.DataFrame(trace)


### Create completed versions

**What:** Fit median, scaled KNN and iterative point imputers to the same inference cohort.

**Why:** Distance-based methods need comparable scales, and methods need comparable samples.

**Your turn:** Run. These completed datasets belong only to inference, not held-out prediction.

**Look for:** No missing cells in each completed version; observed values remain exactly unchanged.

**Interpretation:** An imputed value is a model-based replacement, not a recovered known measurement.


In [ ]:
if framingham is not None:
    fr_base=prepare_framingham(framingham)
    fr_completed=framingham_point_completions(fr_base)
    fr_glucose_missing=fr_base.glucose.isna()
    for name,data in fr_completed.items():
        data.to_csv(OUT/f'framingham_completed_{name}.csv',index=False)
    framingham_summary.update(inference_n=len(fr_base),
        inference_glucose_missing=int(fr_glucose_missing.sum()))


### Generate stochastic PMM/MICE draws

**What:** Retain M completed datasets and a sampler trace.

**Why:** Formal MI must represent uncertainty between completions.

**Your turn:** Run. Inspect the trace; increasing M alone cannot repair a poor imputation model.

**Look for:** Donor values preserve observed support and observed entries remain unchanged.

**Interpretation:** The working conditional models, positivity and MAR assumptions still need justification; this is not a universal imputation recipe.


In [ ]:
if framingham is not None:
    fr_draws,fr_trace=framingham_mi(fr_base)
    table(fr_trace,'framingham_mi_trace')
    plt.plot(fr_trace.draw,fr_trace.glucose_coefficient,'o-',color=PALETTE['methods'])
    plt.xlabel('Retained draw'); plt.ylabel('Log odds ratio per 10 glucose units')
    plt.title('Framingham PMM/MICE · inspect stability, not proof of convergence')
    savefig('framingham_mi_trace')


## 5 · STRESS-TEST — compare the filled data and the conclusion

Because the real missing glucose values are unknown, we cannot score the methods against hidden truth.

Instead we ask:

- Do the filled glucose values look plausible?
- Do the methods produce very different distributions?
- Does the glucose association in the downstream model change?
- What happens if missing glucose was systematically lower or higher than assumed?

This is the real-world version of the **EvidenceLab Triangulation Check**.


### ⚙️ Supporting code: compare distributions without hidden truth

This cell prepares reusable functions used in the analysis that follows.

**▶ Run the cell to prepare the next step.**


In [ ]:
def compare_framingham_distributions(base,completed,draws):
    missing=base.glucose.isna()
    if not missing.any():
        print('This file has no missing glucose; no missing-glucose distribution exists.')
        return pd.DataFrame()
    records=[]; summary=[]
    methods=dict(completed); methods['PMM/MICE draw 1']=draws[0]
    for name,data in methods.items():
        values=data.loc[missing,'glucose']
        records.append(pd.DataFrame({'Method':name,'Glucose':values}))
        summary.append({'method':name,'n_imputed':len(values),'mean':values.mean(),
             'sd':values.std(),'min':values.min(),'max':values.max(),
             'nonpositive':int((values<=0).sum()),
             'outside_observed_range':int(((values<base.glucose.min())|(values>base.glucose.max())).sum())})
    long=pd.concat(records,ignore_index=True)
    # Observed and missing populations may differ; observed is not the hidden truth.
    observed=pd.DataFrame({'Method':'Observed (reference)','Glucose':base.loc[~missing,'glucose']})
    fig=px.violin(pd.concat([observed,long]),x='Method',y='Glucose',color='Method',box=True,
          points=False,title='Framingham · observed and imputed glucose',
          color_discrete_sequence=[PALETTE['observed'],PALETTE['methods'],'#9654C9','#5D3BB5',PALETTE['robust']])
    interactive(fig,'framingham_imputed_distributions')
    table(pd.DataFrame(summary),'framingham_plausibility')
    long.to_csv(OUT/'framingham_imputed_glucose.csv',index=False)
    return pd.DataFrame(summary)


### Inspect the filled-value distributions

**What:** Compare spread, concentration and range for the originally missing cells.

**Why:** A smooth or narrow distribution is not automatically better.

**Your turn:** Run. Look for a pile-up at the median and compare the donor draw.

**Look for:** Plausibility flags and method-specific violin plots.

**Interpretation:** Natural missing glucose has no known truth. You cannot select a winner using hidden-value RMSE here.


In [ ]:
if framingham is not None:
    fr_plausibility=compare_framingham_distributions(fr_base,fr_completed,fr_draws)


#### 🔎 What does this mean?
Because the true missing glucose values are unknown, compare whether the completed distributions are believable rather than declaring one method correct.


### ⚙️ Supporting code: compare the same association model

This cell prepares reusable functions used in the analysis that follows.

**▶ Run the cell to prepare the next step.**


In [ ]:
def compare_framingham_models(base,completed,draws):
    rows=[]
    cc=fit_framingham(base.dropna(subset=['glucose']))
    low,high=cc.conf_int().loc['glucose10']
    rows.append(dict(method='Complete case',estimate=np.exp(cc.params['glucose10']),
         low=np.exp(low),high=np.exp(high),n=int(cc.nobs),interval_kind='Complete-case model'))
    for name,data in completed.items():
        fit=fit_framingham(data)
        rows.append(dict(method=name+' (point only)',estimate=np.exp(fit.params['glucose10']),
             low=np.nan,high=np.nan,n=int(fit.nobs),interval_kind='Single imputation: no MI interval'))
    fits=[fit_framingham(data) for data in draws]
    pooled_result,covariance=rubin_pool(fits)
    pooled_result.to_csv(OUT/'framingham_rubin_pool_log_odds.csv',index=False)
    np.savez_compressed(OUT/'framingham_mi_model_draws.npz',
        estimates=np.array([fit.params.to_numpy() for fit in fits]),
        covariances=np.array([fit.cov_params().to_numpy() for fit in fits]),
        completed=np.array([data.to_numpy() for data in draws]),columns=np.array(base.columns,dtype=str))
    row=pooled_result.set_index('term').loc['glucose10']
    rows.append(dict(method='PMM/MICE pooled',estimate=np.exp(row.estimate),low=np.exp(row.low),
        high=np.exp(row.high),n=len(base),interval_kind='Rubin pooled'))
    result=pd.DataFrame(rows)
    table(result,'framingham_model_comparison')
    method_forest(result,'framingham_model_comparison','Same model · compare handling approaches')
    return result,pooled_result


### Compare downstream estimates and justified intervals

**What:** Fit the same adjusted association model after each method.

**Why:** Agreement about filled values and agreement about the model are different checks.

**Your turn:** Run. Compare sample sizes, ORs and the interval labels.

**Look for:** Complete-case and pooled-MI intervals; single-imputation points without misleading MI intervals.

**Interpretation:** Ordinary single-completion standard errors ignore filling uncertainty. The pooled interval is conditional on the working models and assumptions.


In [ ]:
if framingham is not None:
    fr_comparison,fr_pool=compare_framingham_models(fr_base,fr_completed,fr_draws)
    framingham_summary['method_OR_range']=[float(fr_comparison.estimate.min()),float(fr_comparison.estimate.max())]
    framingham_summary['pooled_glucose10_log_OR']=fr_pool.set_index('term').loc['glucose10'].to_dict()


#### 🔎 What does this mean?
Now compare the same scientific question after different handling methods. Large changes would warn that the conclusion is sensitive to the missing-data choice.


### ⚙️ Supporting code: stress-test unrecorded glucose

This cell prepares reusable functions used in the analysis that follows.

**▶ Run the cell to prepare the next step.**


In [ ]:
def stress_test_framingham(base,draws,deltas):
    missing=base.glucose.isna(); rows=[]
    for delta in deltas:
        fits=[]
        for original in draws:
            adjusted=original.copy(deep=True)
            adjusted.loc[missing,'glucose']+=delta
            assert np.array_equal(adjusted.loc[~missing,'glucose'],original.loc[~missing,'glucose'])
            if (adjusted.glucose<=0).any():
                raise ValueError('A chosen delta creates nonpositive glucose. Reconsider the scenario; values were not silently clipped.')
            fits.append(fit_framingham(adjusted))
        pooled_result,_=rubin_pool(fits)
        r=pooled_result.set_index('term').loc['glucose10']
        rows.append(dict(method='MAR working model' if delta==0 else f'Glucose delta {delta:+g}',
            delta=delta,estimate=np.exp(r.estimate),low=np.exp(r.low),high=np.exp(r.high),
            interval_kind='Rubin pooled'))
    result=pd.DataFrame(rows)
    table(result,'framingham_delta_sensitivity')
    method_forest(result,'framingham_delta_sensitivity','Would another glucose assumption change the conclusion?')
    return result


### Change the unseen-value assumption

**What:** Shift only imputed glucose, then refit and pool each scenario.

**Why:** Comparing three MAR-style methods does not investigate every plausible MNAR departure.

**Your turn:** Run the illustrative deltas. Later justify a different range and rerun.

**Look for:** Which pooled intervals include the null OR=1 and how the estimates change.

**Interpretation:** These offsets use the CSV’s recorded glucose units; they are not clinically calibrated. Stability across these scenarios does not prove the assumptions.


In [ ]:
FR_DELTAS = [-20.,-10.,0.,10.,20.]
if framingham is not None:
    fr_sensitivity=stress_test_framingham(fr_base,fr_draws,FR_DELTAS)
    fr_sensitivity['interval_includes_one']=(fr_sensitivity.low<=1)&(fr_sensitivity.high>=1)
    display(fr_sensitivity[['method','estimate','low','high','interval_includes_one']])
    framingham_summary['sensitivity_intervals_include_one']=fr_sensitivity.interval_includes_one.tolist()


#### 🔎 What does this mean?
This is the key robustness check: does the conclusion survive when we deliberately make the missing glucose values lower or higher?


## 6 · MODEL — start prediction again from the raw data

The completed datasets above were created for the **inference exercise**.

For machine learning, we start again from the original incomplete data.

Why? Because prediction needs a clean separation between training and test data.

✅ **Split first → fit the imputer inside training → apply it to test**

> **The test set must not teach the imputer.**

The technical engine below builds this safe workflow.


### ⚙️ Supporting code: leakage-safe Framingham prediction

This cell prepares reusable functions used in the analysis that follows.

**▶ Run the cell to prepare the next step.**


In [ ]:
def predict_framingham(raw):
    data=raw.dropna(subset=['TenYearCHD']).copy()
    numeric=['age','sysBP','diaBP','BMI','heartRate','glucose']
    categorical=['male','diabetes','currentSmoker']
    X=data[numeric+categorical]; y=data.TenYearCHD
    Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=SEED,stratify=y)
    assert set(Xtr.index).isdisjoint(Xte.index)
    imputers={'Median':SimpleImputer(strategy='median'),
        'KNN':KNNImputer(n_neighbors=5,weights='distance'),
        'Iterative':IterativeImputer(random_state=SEED,max_iter=30,tol=1e-3)}
    rows=[]
    splits=list(StratifiedKFold(4,shuffle=True,random_state=SEED).split(Xtr,ytr))
    for name,imputer in imputers.items():
        pre=ColumnTransformer([
            ('numeric',Pipeline([('scale',StandardScaler()),('impute',imputer)]),numeric),
            ('categorical',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),
                ('encode',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]),categorical)])
        pipeline=Pipeline([('preprocess',pre),('model',LogisticRegression(max_iter=2000))])
        cv=cross_validate(pipeline,Xtr,ytr,cv=splits,scoring='roc_auc',return_estimator=True)
        for (train_idx,_),fitted in zip(splits,cv['estimator']):
            scale=fitted.named_steps['preprocess'].named_transformers_['numeric'].named_steps['scale']
            assert np.allclose(scale.mean_,Xtr.iloc[train_idx][numeric].mean())
        pipeline.fit(Xtr,ytr)
        scale=pipeline.named_steps['preprocess'].named_transformers_['numeric'].named_steps['scale']
        before=scale.mean_.copy(); changed=Xte.copy(); changed['glucose']=1e6
        pipeline.predict_proba(changed)
        assert np.array_equal(before,scale.mean_)
        assert np.allclose(before,Xtr[numeric].mean())
        probabilities=pipeline.predict_proba(Xte)[:,1]
        rows.append(dict(method=name,cv_AUC=float(cv['test_score'].mean()),
            test_AUC=roc_auc_score(yte,probabilities),test_Brier=brier_score_loss(yte,probabilities),
            train_n=len(Xtr),test_n=len(Xte),training_fold_scaling_verified=True,
            test_mutation_state_unchanged=True,outcome_excluded_from_imputer=True))
    result=pd.DataFrame(rows); table(result,'framingham_prediction')
    print('Prespecified training-CV selection:',result.loc[result.cv_AUC.idxmax(),'method'])
    return result


### Evaluate held-out prediction

**What:** Run the three prespecified prediction pipelines.

**Why:** All learned transformations must remain inside the training boundary.

**Your turn:** Run. Read training CV first and avoid tuning on the displayed test results.

**Look for:** Training/test counts, AUC, Brier score and leakage-audit flags.

**Interpretation:** Performance is internal to this teaching split. Missingness patterns, calibration and external populations still require investigation.


In [ ]:
if framingham is not None:
    fr_prediction=predict_framingham(framingham)
    framingham_summary['prediction_training_CV_choice']=fr_prediction.loc[fr_prediction.cv_AUC.idxmax(),'method']


## 7 · INSIGHTS — what did we learn from Framingham?

Do not ask only which method produced the smallest number.

Ask:

- Did the different methods create plausible glucose values?
- Did they tell a similar downstream story?
- Did the conclusion survive the sensitivity scenarios?
- What assumptions would you report to a reader?

**Agreement supports robustness across the approaches we tested. It does not prove that the missingness assumption is true.**


### Write a results-based interpretation

**What:** Generate a concise report from the actual computed values.

**Why:** Interpretation must follow this run, including disagreement.

**Your turn:** Run, then rewrite the report in your own words with a data-collection assumption.

**Look for:** The number of missing values, model range and scenario-specific interval decisions.

**Interpretation:** No method is chosen by unknown hidden-value error. Agreement supports robustness across tested approaches; it does not establish the mechanism.


In [ ]:
if framingham is not None:
    includes=fr_sensitivity.interval_includes_one
    decision=('All tested pooled intervals include OR=1.' if includes.all() else
              'No tested pooled interval includes OR=1.' if not includes.any() else
              'Whether the interval includes OR=1 changes across the tested assumptions.')
    report=(f"### Your Framingham findings\n\n"
        f"- Glucose missing: **{framingham_summary['glucose_missing']:,} / {len(framingham):,} "
        f"({framingham_summary['glucose_missing_pct']:.2f}%)**. The mechanism remains unknown.\n"
        f"- Across handling approaches, adjusted OR estimates per ten glucose units range from "
        f"**{fr_comparison.estimate.min():.3f} to {fr_comparison.estimate.max():.3f}**.\n"
        f"- **{decision}** These illustrative deltas do not cover all possible MNAR processes.\n"
        f"- Single-imputation points omit filling uncertainty; formal MI results use Rubin pooling.\n"
        f"- The prediction exercise restarted from raw data with training-only preprocessing.")
    display(Markdown(report)); (OUT/'framingham_interpretation.md').write_text(report)
else:
    print('Track 2 status:',framingham_summary['status'],'— no real-world results were invented.')


## Finish · your EvidenceLab checklist

When you encounter missing data in your own work:

1. **MAP:** How much is missing, and where?
2. **QUESTION:** Is missingness associated with information you observed?
3. **MECHANISM:** What missingness explanation is plausible?
4. **METHOD:** Which handling method fits your goal?
5. **STRESS-TEST:** Would another reasonable assumption change the conclusion?
6. **MODEL:** For ML, did you prevent leakage?
7. **INSIGHTS:** Did you report what you assumed and what remained uncertain?

> **Don't ask only: “How much is missing?” Ask: “Why is it missing, and does my conclusion still hold?”**

### Want the technical detail?
The references and exported results remain available below for readers who want to go deeper.


### Save the session and results

**What:** Record versions and calculated summaries.

**Why:** A saved chart needs reproducible numerical provenance.

**Your turn:** Run. Outputs include CSVs, static PNG/SVG figures, interactive HTML and a JSON results record.

**Look for:** The completion message and nonempty session/results files.

**Interpretation:** If Track 2 was skipped or unavailable, its status is explicit; simulation results still export.


In [ ]:
import importlib.metadata as metadata
versions={p:metadata.version(p) for p in ['numpy','pandas','scipy','statsmodels',
    'scikit-learn','matplotlib','plotly']}
versions['python']=sys.version
(OUT/'session.json').write_text(json.dumps(versions,indent=2))
summary={'seed':SEED,'n':N,'m':M,'generating_coefficient':.8,
    'complete_data_estimate':float(truth_fit.params['biomarker']),
    'framingham':framingham_summary,'delta_all_intervals_above_zero':bool(positive),
    'delta_estimate_range':[float(sensitivity.estimate.min()),float(sensitivity.estimate.max())],
    'rubin_MAR_biomarker':pooled['MAR'].set_index('term').loc['biomarker'].to_dict()}
(OUT/'results.json').write_text(json.dumps(summary,indent=2))
print('Finished: both selected tracks and reproducibility exports completed.')
display(versions)
